# Clase 172 — Entrenamiento multi-dispositivo (tf.distribute)

Escalar el training a varias GPUs (y nodos). Estrategias TF: **MirroredStrategy** (1 nodo,
varias GPUs), **MultiWorkerMirroredStrategy** (varios nodos), **TPUStrategy**. Equivalentes
PyTorch: **DDP** y **FSDP** (estándar para LLMs grandes).

Requiere: `tensorflow` (opcional). El código es correcto; sin múltiples GPUs no se ejecuta aquí.

## 1. MirroredStrategy: data parallelism en un nodo

Cada GPU tiene una **réplica** del modelo, procesa un mini-batch distinto, y los gradientes se
promedian con **all-reduce**. Todo se construye dentro de `strategy.scope()`.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    TF_OK = True
except Exception:
    TF_OK = False
    print("tensorflow no instalado -> se muestra la API (no se ejecuta)")

if TF_OK:
    strategy = tf.distribute.MirroredStrategy()
    print("replicas en sync:", strategy.num_replicas_in_sync)
    with strategy.scope():
        model = keras.Sequential([
            keras.Input(shape=(784,)),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ])
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                      metrics=["accuracy"])
    print("modelo replicado en todas las GPUs del nodo")
else:
    print("strategy = tf.distribute.MirroredStrategy()")
    print("with strategy.scope(): model = build(); model.compile(...)")

## 2. Batch global y la regla de escalado del learning rate

Con `N` réplicas y `batch_size` por réplica, el **batch global** es `N × batch_size`. La regla
lineal ajusta el LR proporcionalmente.

In [ ]:
import numpy as np
num_replicas = 4
per_replica_batch = 32
global_batch = num_replicas * per_replica_batch
base_lr = 1e-3
scaled_lr = base_lr * num_replicas                 # linear scaling rule

print(f"batch por replica: {per_replica_batch}")
print(f"batch global:      {global_batch}")
print(f"LR escalado:       {scaled_lr:.4f}  (= base_lr x num_replicas)")

## 3. Multi-nodo: MultiWorkerMirroredStrategy

Para varios nodos se usa `MultiWorkerMirroredStrategy` y la variable de entorno `TF_CONFIG`
(orquestada por K8s, Slurm o Vertex AI) que define cluster y rol de cada worker.

In [ ]:
import json, os
tf_config = {
    "cluster": {"worker": ["host1:12345", "host2:12345"]},
    "task": {"type": "worker", "index": 0},        # cada nodo cambia su index
}
print("TF_CONFIG =", json.dumps(tf_config["cluster"]))

if TF_OK:
    # os.environ["TF_CONFIG"] = json.dumps(tf_config)   # lo setea el orquestador
    print("strategy = tf.distribute.MultiWorkerMirroredStrategy()  # lee TF_CONFIG")
else:
    print("MultiWorkerMirroredStrategy + TF_CONFIG (cluster + task por nodo)")

## 4. Gradient accumulation y model parallelism

- **Gradient accumulation**: acumular gradientes de varios mini-batches antes de aplicar el
  update → simula un batch grande en GPUs con poca VRAM.
- **Model/pipeline parallelism**: el modelo se **divide** entre GPUs (para modelos que no caben
  en una sola). **FSDP** (PyTorch) / **DeepSpeed ZeRO** shardean parámetros, gradientes y estados
  del optimizer — el estándar para entrenar LLMs.

## 5. Equivalentes PyTorch

```bash
# DDP: data parallelism, un proceso por GPU
torchrun --nproc_per_node=4 train.py
```
```python
from torch.nn.parallel import DistributedDataParallel as DDP
model = DDP(model)                     # data parallel

# PyTorch Lightning abstrae todo con un kwarg:
trainer = L.Trainer(strategy="ddp", devices=4)      # o strategy="fsdp"
```

FSDP/DeepSpeed permiten Llama 70B en 8× H100 (640 GB) shardeando el modelo.

## 6. Gradient accumulation (esquema ejecutable)

Acumular gradientes de varios mini-batches antes de aplicar el update simula un batch grande en
GPUs con poca VRAM. Aquí simulamos la lógica con numpy (promediar gradientes acumulados).

In [ ]:
import numpy as np
np.random.seed(0)
accum_steps = 4
grads = [np.random.randn(3) for _ in range(accum_steps)]   # 4 mini-batches
accumulated = np.zeros(3)
for i, g in enumerate(grads):
    accumulated += g / accum_steps          # promedio: batch efectivo = 4x
    if (i + 1) % accum_steps == 0:
        print("aplicar update con gradiente acumulado:", np.round(accumulated, 3))
        accumulated = np.zeros(3)

## 7. PyTorch DDP (guarded)

In [ ]:
try:
    import torch
    from torch.nn.parallel import DistributedDataParallel as DDP
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
    # model = DDP(model.to(rank), device_ids=[rank])   # 1 proceso por GPU
    print("lanzar con: torchrun --nproc_per_node=4 train.py")
except Exception as e:
    print("torch no instalado:", type(e).__name__)
    print("DDP: torchrun --nproc_per_node=4 train.py ; model = DDP(model)")
    print("FSDP para modelos que no caben en 1 GPU (shard de parametros)")

## Ejercicios

1. Entrenar el mismo modelo en single-GPU vs `MirroredStrategy` y comparar el wall-time por época.
2. Aplicar la regla de escalado lineal del LR con el batch global y verificar convergencia similar.
3. Implementar gradient accumulation manual (4 mini-batches de 128 → batch efectivo 512).
4. Reescribir un training loop con PyTorch Lightning y `strategy='ddp', devices=4`.

## Conclusiones

- `MirroredStrategy` da data parallelism en un nodo: réplicas + all-reduce de gradientes.
- El batch global crece con el número de réplicas; hay que escalar el LR (regla lineal).
- Multi-nodo usa `MultiWorkerMirroredStrategy` + `TF_CONFIG`, orquestado por K8s/Slurm/Vertex.
- Cuando el modelo no cabe en una GPU se usa FSDP/DeepSpeed (sharding); Lightning lo abstrae.